# DPO baseline (optional)

Continue training the SFT adapter with Direct Preference Optimization
as a preference-learning baseline to compare against GRPO. The
original plan was to build chosen/rejected pairs by sampling from
the SFT model and labeling them with our verifiable checker, but
that turned out to be too slow in Colab — see the note in the
rejection-sampling cell below. We fall back to a public DPO
dataset instead.

In [1]:
# Config. Continue from the SFT adapter, not the base model — that's
# the whole point. N_SAMPLES_PER_PROBLEM was supposed to be 4 but we
# lowered it because generation in Colab is too slow.
!pip install -q -U bitsandbytes transformers trl peft accelerate datasets torchao

# Mount Drive and restore data
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE = '/content/drive/MyDrive/llm_posttraining'

# Restore data
if not os.path.exists('/content/data/sft_train'):
    shutil.copytree(f'{DRIVE}/data/sft_train',    '/content/data/sft_train')
    shutil.copytree(f'{DRIVE}/data/sft_val',      '/content/data/sft_val')
    shutil.copytree(f'{DRIVE}/data/sft_test',     '/content/data/sft_test')
    shutil.copytree(f'{DRIVE}/data/grpo_prompts', '/content/data/grpo_prompts')

# Restore SFT checkpoint
if not os.path.exists('/content/checkpoints/sft_final'):
    os.makedirs('/content/checkpoints', exist_ok=True)
    shutil.copytree(f'{DRIVE}/checkpoints/sft_final', '/content/checkpoints/sft_final')

print('All restored.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.8 MB/s eta 0:00:00
Mounted at /content/drive
All restored.


In [6]:
import torch, re
from datasets import load_from_disk, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
from trl import DPOTrainer, DPOConfig
import wandb

SFT_ADAPTER = '/content/checkpoints/sft_final'
BASE_MODEL  = 'Qwen/Qwen3-8B'
HF_REPO_DPO = 'Chaitanya77/qwen3-8b-math-dpo'  # <-- change this
N_SAMPLES_PER_PROBLEM = 2   # sample 4 completions, pick correct vs wrong

In [7]:
# Helpers for parsing \boxed{} and comparing answers. Same logic
# as the GRPO reward function — by construction, the chosen samples
# here are the ones GRPO would assign +1 reward to.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map='auto', torch_dtype=torch.bfloat16
)
sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER)
sft_model.eval()

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [8]:
# Generate completions and label them. Warning: this is the slowest
# cell in the whole pipeline. For 2000 problems × 4 samples it can
# take 2+ hours even on H100. If it stalls, lower the problem count
# or switch to a pre-built DPO dataset (see below).
def extract_boxed(text):
    m = re.search(r'\\boxed\{([^}]*)\}', text)
    return m.group(1).strip() if m else None

def normalize_answer(ans):
    if ans is None: return ''
    return ans.strip().lower().replace(' ', '')

def answers_match(pred, gold):
    return normalize_answer(pred) == normalize_answer(gold)

In [10]:
from datasets import load_dataset

# Free memory and turn the list into a HF Dataset, then split.
# DPO needs paired chosen/rejected so we keep only problems where
# the model produced at least one of each.
dpo_dataset = load_dataset("argilla/dpo-mix-7k", split="train")

# Rename columns to match DPO trainer expected format
dpo_dataset = dpo_dataset.select(range(2000))
dpo_split = dpo_dataset.train_test_split(test_size=0.1, seed=42)
print(f'DPO train: {len(dpo_split["train"])} | val: {len(dpo_split["test"])}')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.43M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/750 [00:00<?, ? examples/s]

DPO train: 1800 | val: 200


In [11]:
# Reload SFT model as the policy to be updated. is_trainable=True
# so PEFT lets gradients flow into the adapter.
del sft_model, base_model
torch.cuda.empty_cache()
import gc; gc.collect()

dpo_dataset = Dataset.from_list(dpo_pairs)
dpo_split   = dpo_dataset.train_test_split(test_size=0.1, seed=42)
print(f'DPO train: {len(dpo_split["train"])} | val: {len(dpo_split["test"])}')

DPO train: 7 | val: 1


In [12]:
# DPO config. beta=0.1 is a standard starting point — higher means
# stay closer to the SFT reference. max_steps=100 is short because
# DPO converges quickly on a small preference set.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map='auto', torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=True)
model.config.use_cache = False

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [14]:
dpo_config = DPOConfig(
    output_dir='/content/checkpoints/dpo',
    beta=0.1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    num_train_epochs=1,
    max_steps=100,
    bf16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_length=2048,
    save_strategy='epoch',
    logging_steps=25,
    report_to='wandb',
    run_name='qwen3-8b-dpo-math',
)

wandb.init(project='verifiable-reasoning-posttraining', name='04_dpo')

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # None = use SFT adapter as implicit reference (PEFT handles this)
    args=dpo_config,
    train_dataset=dpo_split['train'],
    eval_dataset=dpo_split['test'],
    processing_class=tokenizer,
)
dpo_trainer.train()
wandb.finish()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Adding EOS to train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
25,0.132000
50,0.000000
75,0.000000
100,0.000000


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offl

train/entropy,█▁▂▂
train/epoch,▁▃▆██
train/global_step,▁▃▆██
train/grad_norm,█▄▂▁
train/learning_rate,█▅▂▁
train/logits/chosen,█▁▁▁
train/logits/rejected,█▂▁▂
train/logps/chosen,▁█▆█
train/logps/rejected,█▁▁▁
train/loss,█▁▁▁
+6,...


In [16]:
from huggingface_hub import login
login()

In [17]:
dpo_trainer.model.save_pretrained('/content/checkpoints/dpo_final')
dpo_trainer.model.push_to_hub(HF_REPO_DPO)
print('DPO adapter saved and pushed.')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 46.3kB / 87.4MB            

  ...adapter_model.safetensors:  78%|#######7  |  136MB /  175MB            

DPO adapter saved and pushed.


In [19]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, os
DRIVE = '/content/drive/MyDrive/llm_posttraining'

# Save DPO checkpoint only
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
shutil.copytree('/content/checkpoints/dpo_final',
                f'{DRIVE}/checkpoints/dpo_final',
                dirs_exist_ok=True)

print('DPO checkpoint saved to Drive.')

Mounted at /content/drive
DPO checkpoint saved to Drive.
